# Ops smoke test: does the Colab job queue work end to end, and what hardware is this?

Run by the Colab job worker (`notebooks/05-ops/colab_worker.ipynb`) as the first job, or by hand. It trains nothing and reads no data. It records the runtime (Python, GPU name and memory, PyTorch and CUDA if present), probes the data stack the pilots need (TensorFlow and TensorFlow Datasets: versions, whether `tfds.load` exists, the error chain if not; on Colab it first re-runs the pilots' `pip install` in a subprocess and probes again), checks that Drive is mounted and writable, does a small GPU operation if a GPU is present, and writes a run directory `ops_smoke_<UTC>` with `reports/acceptance_report.json`, `reports/run_summary.md` and `COMPLETE`, so that the worker's result bundle and a human both have something to read. See `docs/plans/colab-handoff.md` §5.

In [ ]:
# ================================================================
# 0. Package
# ================================================================
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/ROMUSKING/para_001'
REPO_REF = 'main'  # branch, tag or commit SHA. The job worker pins this to the job's commit.
REPO_DIR = '/content/para_001' if IN_COLAB else str(Path.cwd().resolve().parents[1])

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--all'], check=False)
    for candidate in (f'origin/{REPO_REF}', REPO_REF):
        resolved = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--verify', '--quiet', candidate + '^{commit}'],
                                  capture_output=True, text=True)
        if resolved.returncode == 0:
            subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--quiet', '--detach', resolved.stdout.strip()], check=True)
            break
    else:
        raise RuntimeError(f'Cannot resolve {REPO_REF!r}.')
REPO_COMMIT = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=False)
    RUN_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production/runs')
else:
    RUN_ROOT = Path.home() / 'adjointrwm_runs'
print('adjointrwm at', REPO_COMMIT, '| runs ->', RUN_ROOT)

In [ ]:
# ================================================================
# 1. Record the runtime and write the run directory
# ================================================================
import datetime
import json
import platform
import time

from adjointrwm.io import atomic_write_json, atomic_write_text

RUN_ID = 'ops_smoke_' + datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = RUN_ROOT / RUN_ID
if RUN_DIR.exists():
    raise RuntimeError(f'{RUN_DIR} exists; run directories are immutable.')
for sub in ('config', 'reports'):
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)


def shell(command):
    try:
        done = subprocess.run(command, capture_output=True, text=True)
    except OSError:     # no such program (for example nvidia-smi on a CPU runtime)
        return ''
    return done.stdout.strip() if done.returncode == 0 else ''


gpu = shell(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'])
ENV = {'python': sys.version.split()[0], 'platform': platform.platform(), 'in_colab': IN_COLAB, 'repo_commit': REPO_COMMIT,
       'nvidia_smi': gpu or None, 'cpu_count': os.cpu_count()}
try:
    import torch

    ENV['torch'] = torch.__version__
    ENV['cuda_available'] = bool(torch.cuda.is_available())
    if torch.cuda.is_available():
        ENV['cuda_device'] = torch.cuda.get_device_name(0)
        ENV['cuda_memory_gib'] = round(torch.cuda.get_device_properties(0).total_memory / 1024 ** 3, 2)
        started = time.perf_counter()
        x = torch.randn(2048, 2048, device='cuda')
        torch.cuda.synchronize()
        ENV['gpu_matmul_ok'] = bool(torch.isfinite((x @ x).sum()).item())
        ENV['gpu_matmul_seconds'] = round(time.perf_counter() - started, 4)
except ImportError:
    ENV['torch'] = None
_PROBE = """
import json, os, sys, traceback
from importlib import metadata
out = {'executable': sys.executable, 'cwd': os.getcwd(), 'cwd_entries_with_tensorflow': sorted(n for n in os.listdir('.') if 'tensorflow' in n.lower())}
versions = {}
for name in ('tensorflow', 'tensorflow-datasets', 'tfds-nightly', 'numpy', 'protobuf', 'etils', 'array_record', 'dm-tree', 'rlds', 'pyarrow', 'absl-py'):
    try:
        versions[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        versions[name] = None
out['versions'] = versions
try:
    import tensorflow as tf
    out['tensorflow_import'] = 'ok ' + tf.__version__
except BaseException as error:
    out['tensorflow_import'] = type(error).__name__ + ': ' + str(error)[:300]
try:
    import tensorflow_datasets as tfds
    out['tfds_file'] = getattr(tfds, '__file__', None)
    out['tfds_path'] = [str(p) for p in getattr(tfds, '__path__', [])]
    out['tfds_has_load'] = hasattr(tfds, 'load')
    if not out['tfds_has_load']:
        out['tfds_public_names'] = sorted(n for n in dir(tfds) if not n.startswith('_'))[:40]
        try:
            tfds.load
        except BaseException as error:
            chain, e = [], error
            while e is not None and len(chain) < 4:
                chain.append(type(e).__name__ + ': ' + str(e)[:300])
                e = e.__cause__ or e.__context__
            out['tfds_load_error_chain'] = chain
            out['tfds_load_traceback_tail'] = ''.join(traceback.format_exception(error))[-1500:]
except BaseException as error:
    out['tfds_import'] = type(error).__name__ + ': ' + str(error)[:300]
print(json.dumps(out))
"""


def probe_data_stack():
    done = subprocess.run([sys.executable, '-c', _PROBE], capture_output=True, text=True, timeout=600)
    lines = done.stdout.strip().splitlines()
    try:
        return json.loads(lines[-1])
    except (IndexError, json.JSONDecodeError):
        return {'probe_failed': True, 'returncode': done.returncode, 'stderr_tail': done.stderr[-800:]}


DATA_STACK = {'before': probe_data_stack()}
if IN_COLAB:    # the pilots' own install line, run in a subprocess of the same interpreter, then the same probe again
    installed = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'tensorflow-datasets>=4.9', 'rlds', 'pyarrow'],
                               capture_output=True, text=True, timeout=1500)
    DATA_STACK['pip_returncode'] = installed.returncode
    DATA_STACK['pip_output_tail'] = (installed.stdout + installed.stderr)[-1200:]
    DATA_STACK['after_pip'] = probe_data_stack()
ENV['data_stack'] = DATA_STACK
ENV['drive_run_root_writable'] = bool(os.access(RUN_ROOT, os.W_OK))
atomic_write_json(RUN_DIR / 'config' / 'environment.json', ENV)
REPORT = {'run_id': RUN_ID, 'milestone': 'OPS smoke (job queue check)', 'repo_commit': REPO_COMMIT, 'environment': ENV,
          'passed': bool(ENV['drive_run_root_writable'] and (not ENV.get('cuda_available') or ENV.get('gpu_matmul_ok'))),
          'claim_boundary': 'Checks the runtime and the job queue only. It trains nothing and measures nothing about the research question.'}
atomic_write_json(RUN_DIR / 'reports' / 'acceptance_report.json', REPORT)
atomic_write_text(RUN_DIR / 'reports' / 'run_summary.md', '\n'.join([
    f'# Ops smoke `{RUN_ID}`', '', f"Passed: **{REPORT['passed']}**", '', f"- GPU: {ENV.get('cuda_device') or gpu or 'none'}",
    f"- Memory (GiB): {ENV.get('cuda_memory_gib')}", f"- PyTorch: {ENV.get('torch')} (CUDA available: {ENV.get('cuda_available')})",
    f"- Python: {ENV['python']} on {ENV['platform']}", f"- Repository commit: `{REPO_COMMIT}`", f"- In Colab: {IN_COLAB}; Drive writable: {ENV['drive_run_root_writable']}", f"- tensorflow-datasets has `load`: {(DATA_STACK.get('after_pip') or DATA_STACK['before']).get('tfds_has_load')} (details in `config/environment.json`)", '']))
atomic_write_text(RUN_DIR / 'COMPLETE', datetime.datetime.now(datetime.UTC).isoformat() + '\n')
print(json.dumps(ENV, indent=2))
print('Run directory:', RUN_DIR)